
 Cargar el CSV limpio

In [8]:
import pandas as pd
import numpy as np

df = pd.read_csv("secop_ii_agencia_logistica_limpio.csv")

cols_fecha = [c for c in df.columns if c.startswith('fecha') or c == 'ultima_actualizacion']
for c in cols_fecha:
    df[c] = pd.to_datetime(df[c], errors='coerce')

print(df.shape)

(8051, 88)


Variables de tiempo

In [9]:
df['anio_firma'] = df['fecha_de_firma'].dt.year
df['mes_firma'] = df['fecha_de_firma'].dt.month
df['trimestre_firma'] = df['fecha_de_firma'].dt.quarter
df['nombre_mes_firma'] = df['fecha_de_firma'].dt.month_name()

Tratamiento del valor del contrato

In [10]:
df['valor_log'] = np.log1p(df['valor_del_contrato'].clip(lower=0))

q1, q3 = df['valor_log'].quantile([0.25, 0.75])
iqr = q3 - q1
limite_superior = q3 + 1.5 * iqr
df['es_outlier_valor'] = df['valor_log'] > limite_superior

p995 = df['valor_del_contrato'].quantile(0.995)
df['valor_contrato_capped'] = df['valor_del_contrato'].clip(upper=p995)

Agregar periodo_mensual 

In [11]:
df['periodo_mensual'] = df['fecha_de_firma'].dt.to_period('M').astype(str)

In [12]:
print(df.shape)
df[['fecha_de_firma', 'anio_firma', 'mes_firma', 'nombre_mes_firma', 'trimestre_firma', 'periodo_mensual', 'valor_contrato_capped']].head()

(8051, 96)


,fecha_de_firma,anio_firma,mes_firma,nombre_mes_firma,trimestre_firma,periodo_mensual,valor_contrato_capped
0,2026-07-31,2026.0,7.0,July,3.0,2026-07,7.500000e+07
1,2018-08-06,2018.0,8.0,August,3.0,2018-08,5.848992e+06
2,2021-07-07,2021.0,7.0,July,3.0,2021-07,1.392300e+06
3,2025-10-22,2025.0,10.0,October,4.0,2025-10,3.420000e+07
4,2021-02-25,2021.0,2.0,February,1.0,2021-02,1.149600e+10


In [13]:
RUTA_SALIDA = "secop_ii_agencia_logistica_features_p3.csv"
df.to_csv(RUTA_SALIDA, index=False)
print(f"Guardado: {RUTA_SALIDA} ({df.shape[0]:,} filas x {df.shape[1]} columnas)")

Guardado: secop_ii_agencia_logistica_features_p3.csv (8,051 filas x 96 columnas)
